# 🚀 RenderFast — Distributed Blender GPU Renderer
**Auto-generated by [RenderFast](https://github.com/nepal143/RenderFast) — do not edit the env-var cells manually.**

### Workflow
1. **Cell 2** — Install system dependencies (run once)
2. **Cell 3** — Install Blender (choose version, cached after first run)
3. **Cell 4** — Check Kaggle GPU quota remaining
4. **Cell 5** — Auto-detect your `.blend` file
5. **Cell 6** — ⚙️ **Configure all render settings here**
6. **Cell 7** — Version manager (view history, pick re-render frames)
7. **Cell 8** — 🎬 Run the render (GPU accelerated)
8. **Cell 9** — Package frames into ZIP
9. **Cell 10** — Download links

In [ ]:
# ── Cell 2: System Dependencies ──────────────────────────────────────────────
# Run once per session. Safe to re-run.
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-500:])
    return result.returncode == 0

print("Installing system dependencies...")
run("apt-get update -qq")
run("""apt-get install -y -qq \
    xvfb libxi6 libgconf-2-4 libxrender1 libxxf86vm1 \
    libxfixes3 libgl1 libglu1-mesa libsm6 libice6 \
    libegl1 libx11-6 libxext6 libxrandr2 libwayland-client0 \
    libwayland-egl1 libdbus-1-3 wget tar""")

print("Installing Python dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "kaggle", "requests", "Pillow"], check=True)

print("✅ Done installing system dependencies.")

In [ ]:
# ── Cell 3: Blender Installation ─────────────────────────────────────────────
# Change BLENDER_VERSION to any official release. Cached after first run.
import os, subprocess, pathlib

BLENDER_VERSION = os.environ.get("BLENDER_VERSION", "4.1.1")   # ← override via env var

BLENDER_MAJOR  = ".".join(BLENDER_VERSION.split(".")[:2])       # e.g. 4.1
BLENDER_TAR    = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL    = f"https://download.blender.org/release/Blender{BLENDER_MAJOR}/{BLENDER_TAR}"
INSTALL_DIR    = "/opt/blender"
BLENDER_BIN    = os.path.join(INSTALL_DIR, "blender")

# Persistent cache: store in /kaggle/working so it survives version commits
CACHE_DIR  = pathlib.Path("/kaggle/working/.blender_cache")
CACHE_DIR.mkdir(exist_ok=True)
CACHED_BIN = CACHE_DIR / BLENDER_VERSION / "blender"

if CACHED_BIN.exists():
    BLENDER_BIN = str(CACHED_BIN)
    print(f"✅ Blender {BLENDER_VERSION} loaded from cache: {BLENDER_BIN}")
else:
    tmp = f"/tmp/{BLENDER_TAR}"
    print(f"⬇️  Downloading Blender {BLENDER_VERSION}...")
    subprocess.run(["wget", "-q", "--show-progress", BLENDER_URL, "-O", tmp], check=True)
    print("📦 Extracting...")
    extract_dir = CACHE_DIR / BLENDER_VERSION
    extract_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", tmp, "-C", str(extract_dir), "--strip-components=1"], check=True)
    os.remove(tmp)
    BLENDER_BIN = str(CACHED_BIN)
    print(f"✅ Blender installed at: {BLENDER_BIN}")

result = subprocess.run([BLENDER_BIN, "--version"], capture_output=True, text=True)
print(result.stdout.splitlines()[0])

In [ ]:
# ── Cell 4: Kaggle GPU Quota Check ───────────────────────────────────────────
# Fetches remaining GPU hours directly from the Kaggle API — no manual math.
import json, os, requests
from pathlib import Path
from datetime import datetime, timezone

KAGGLE_CREDS = None
for p in [Path("/root/.kaggle/kaggle.json"), Path.home() / ".kaggle/kaggle.json"]:
    if p.exists():
        KAGGLE_CREDS = json.loads(p.read_text())
        break

# Also check env vars (set by RenderFast backend)
if not KAGGLE_CREDS:
    u = os.environ.get("KAGGLE_USERNAME")
    k = os.environ.get("KAGGLE_KEY")
    if u and k:
        KAGGLE_CREDS = {"username": u, "key": k}

if not KAGGLE_CREDS:
    print("⚠️  No Kaggle credentials found. Add kaggle.json or set env vars.")
else:
    username = KAGGLE_CREDS["username"]
    key      = KAGGLE_CREDS["key"]
    auth     = (username, key)

    # ── GPU quota ────────────────────────────────────────────────────────────
    quota_url = f"https://www.kaggle.com/api/v1/users/{username}/kernel-quota"
    resp = requests.get(quota_url, auth=auth, timeout=15)

    if resp.status_code == 200:
        data = resp.json()
        total_sec   = data.get("gpuQuota",    30 * 3600)   # fallback 30 h
        used_sec    = data.get("gpuUsed",     0)
        reset_ts    = data.get("nextReset",   None)
        remaining_h = (total_sec - used_sec) / 3600
        used_h      = used_sec / 3600
        total_h     = total_sec / 3600
        pct_used    = (used_sec / total_sec * 100) if total_sec else 0

        bar_filled  = int(pct_used / 5)
        bar         = "█" * bar_filled + "░" * (20 - bar_filled)

        print(f"👤 Account      : {username}")
        print(f"⏱️  GPU Used     : {used_h:.2f} h / {total_h:.0f} h  [{bar}] {pct_used:.1f}%")
        print(f"✅ Remaining    : {remaining_h:.2f} hours")
        if reset_ts:
            reset_dt = datetime.fromtimestamp(reset_ts / 1000, tz=timezone.utc)
            print(f"🔄 Quota resets : {reset_dt.strftime('%Y-%m-%d %H:%M UTC')}")
    else:
        # Fallback: try the kernels list endpoint to at least confirm auth works
        test = requests.get("https://www.kaggle.com/api/v1/kernels", auth=auth,
                            params={"pageSize": 1}, timeout=10)
        if test.status_code == 200:
            print(f"⚠️  Auth OK for '{username}' but quota endpoint returned {resp.status_code}.")
            print("   Check https://www.kaggle.com/settings/profile for GPU quota manually.")
        else:
            print(f"❌ Kaggle auth failed (status {test.status_code}). Check credentials.")

    # ── Current GPU info ─────────────────────────────────────────────────────
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
    if gpu.returncode == 0:
        print(f"\n🎮 GPU Available: {gpu.stdout.strip()}")
    else:
        print("\n⚠️  No GPU detected — rendering will use CPU (slow).")

In [ ]:
# ── Cell 5: Auto-detect Blend File ───────────────────────────────────────────
import os, pathlib

BLEND_FILENAME = os.environ.get("BLEND_FILENAME", "")   # injected by RenderFast

print("Scanning /kaggle/input/ for blend files...\n")
BLEND_FILE = None

for root, dirs, files in os.walk("/kaggle/input"):
    for fname in files:
        full = os.path.join(root, fname)
        if fname.endswith(".blend") or (
            not os.path.splitext(fname)[1] and os.path.getsize(full) > 1_000_000
        ):
            size_mb = os.path.getsize(full) / 1024 / 1024
            print(f"  Found: {full}  ({size_mb:.1f} MB)")
            if BLEND_FILE is None:
                BLEND_FILE = full

# If a specific filename was requested, prefer it
if BLEND_FILENAME:
    matches = [str(p) for p in pathlib.Path("/kaggle/input").rglob(BLEND_FILENAME)]
    if matches:
        BLEND_FILE = matches[0]

print()
if BLEND_FILE:
    print(f"✅ Using: {BLEND_FILE}")
    print("   (Change BLEND_FILE below in Cell 6 if this is wrong.)")
else:
    print("❌ No blend file found! Make sure the dataset is attached to this notebook.")

In [ ]:
# ── Cell 6: ⚙️ RENDER CONFIGURATION ─────────────────────────────────────────
# ↓↓↓  Edit everything here. Nothing below needs to be touched.  ↓↓↓

import os

# ── Blend file ────────────────────────────────────────────────────────────────
BLEND_FILE_OVERRIDE = ""           # Leave "" to use auto-detected file from Cell 5
                                   # Or set to a full path: "/kaggle/input/myscene/scene.blend"

# ── Frame range ───────────────────────────────────────────────────────────────
FRAME_START    = int(os.environ.get("START_FRAME", 1))
FRAME_END      = int(os.environ.get("END_FRAME",   50))
FRAME_STEP     = 1                 # Render every Nth frame (1 = every frame)

# Re-render specific frames only (leave empty [] to render full range above)
# Example: RERENDER_FRAMES = [12, 45, 103]
RERENDER_FRAMES = []

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_FORMAT  = "PNG"             # PNG | JPEG | EXR | TIFF
OUTPUT_DIR     = "/kaggle/working/renders/"
OUTPUT_QUALITY = 100               # JPEG quality 1-100 (ignored for PNG/EXR)
COLOR_DEPTH    = "8"               # "8" or "16" (PNG/EXR)

# ── Engine ────────────────────────────────────────────────────────────────────
RENDER_ENGINE  = "CYCLES"          # CYCLES | BLENDER_EEVEE_NEXT | BLENDER_EEVEE
DEVICE         = "GPU"             # GPU | CPU

# ── Cycles settings ───────────────────────────────────────────────────────────
SAMPLES            = 128           # Render samples (higher = better quality, slower)
DENOISE            = True          # Enable Intel/OptiX denoising
DENOISE_ALGORITHM  = "OPENIMAGEDENOISE"   # OPENIMAGEDENOISE | OPTIX | NLM
DENOISE_PASS       = "RGB_ALBEDO_NORMAL"  # RGB | RGB_ALBEDO | RGB_ALBEDO_NORMAL
USE_ADAPTIVE_SAMPLING = True       # Stop sampling early when noise threshold is met
ADAPTIVE_THRESHOLD = 0.01          # Noise threshold for adaptive sampling (0 = disabled)
MIN_SAMPLES        = 0             # Minimum samples before adaptive sampling kicks in

# ── Resolution ────────────────────────────────────────────────────────────────
RESOLUTION_X        = 0           # 0 = use scene default
RESOLUTION_Y        = 0           # 0 = use scene default
RESOLUTION_PERCENT  = 100         # Scale percentage (50 = half res, faster preview)

# ── Motion blur ───────────────────────────────────────────────────────────────
MOTION_BLUR = None                 # None = use scene default | True | False

# ── Versioning ────────────────────────────────────────────────────────────────
VERSION_LABEL = ""                 # Optional label for this render run e.g. "test_128spp"
                                   # Leave "" to auto-name as v1, v2, v3...

# ─────────────────────────────────────────────────────────────────────────────
# Apply BLEND_FILE_OVERRIDE if set
if BLEND_FILE_OVERRIDE:
    BLEND_FILE = BLEND_FILE_OVERRIDE

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not BLEND_FILE or not os.path.exists(BLEND_FILE):
    raise FileNotFoundError(
        f"Blend file not found: '{BLEND_FILE}'\n"
        "Run Cell 5 to scan, or set BLEND_FILE_OVERRIDE above."
    )

size_mb = os.path.getsize(BLEND_FILE) / 1024 / 1024
print(f"File       : {BLEND_FILE}  ({size_mb:.1f} MB)")
if RERENDER_FRAMES:
    print(f"Frames     : Re-rendering specific frames: {RERENDER_FRAMES}")
else:
    print(f"Frames     : {FRAME_START} → {FRAME_END}  (step {FRAME_STEP})  "
          f"= {len(range(FRAME_START, FRAME_END + 1, FRAME_STEP))} frames")
print(f"Engine     : {RENDER_ENGINE} on {DEVICE}")
print(f"Samples    : {SAMPLES}  |  Denoise: {DENOISE} ({DENOISE_ALGORITHM})")
print(f"Adaptive   : {USE_ADAPTIVE_SAMPLING}  threshold={ADAPTIVE_THRESHOLD}  min={MIN_SAMPLES}")
print(f"Format     : {OUTPUT_FORMAT}  depth={COLOR_DEPTH}  quality={OUTPUT_QUALITY}")
res_x = RESOLUTION_X or "(scene)"
res_y = RESOLUTION_Y or "(scene)"
print(f"Resolution : {res_x} x {res_y}  @ {RESOLUTION_PERCENT}%")
print(f"Output     : {OUTPUT_DIR}")

In [ ]:
# ── Cell 7: Version Manager ───────────────────────────────────────────────────
# Tracks every render run. Shows history and determines the next version number.
import json, pathlib, datetime

VERSION_FILE = pathlib.Path("/kaggle/working/.render_versions.json")

def load_versions():
    if VERSION_FILE.exists():
        return json.loads(VERSION_FILE.read_text())
    return []

def next_version_label():
    vers = load_versions()
    if VERSION_LABEL:
        return VERSION_LABEL
    return f"v{len(vers) + 1}"

versions = load_versions()
CURRENT_VERSION = next_version_label()

print(f"📋 Render History  ({len(versions)} previous runs)\n")
print(f"{'#':<4} {'Label':<16} {'Frames':<14} {'Samples':<9} {'Engine':<12} {'Date'}")
print("─" * 72)
for i, v in enumerate(versions, 1):
    fr = f"{v.get('start_frame')}–{v.get('end_frame')}"
    if v.get("rerender_frames"):
        fr = f"re:{v['rerender_frames']}"
    print(f"{i:<4} {v.get('label','?'):<16} {fr:<14} {str(v.get('samples','?')):<9} "
          f"{v.get('engine','?'):<12} {v.get('date','?')[:10]}")
print("─" * 72)
print(f"\n▶ Current run will be saved as: '{CURRENT_VERSION}'\n")

# Determine actual frames to render (supports re-render of specific frames)
if RERENDER_FRAMES:
    FRAMES_TO_RENDER = sorted(set(RERENDER_FRAMES))
    print(f"🔁 Re-render mode: rendering frames {FRAMES_TO_RENDER}")
else:
    FRAMES_TO_RENDER = list(range(FRAME_START, FRAME_END + 1, FRAME_STEP))
    print(f"🎬 Full render: {len(FRAMES_TO_RENDER)} frames")

In [ ]:
# ── Cell 8: 🎬 GPU-Accelerated Render ────────────────────────────────────────
import os, subprocess, sys, tempfile, time, datetime

# ── Build the Blender Python setup script (runs inside Blender) ──────────────
GPU_SCRIPT = "/tmp/gpu_render_setup.py"

with open(GPU_SCRIPT, "w") as f:
    f.write(f"""
import bpy, os, sys

scene = bpy.context.scene

# ── Engine ────────────────────────────────────────────────────────────────────
scene.render.engine = "{RENDER_ENGINE}"

# ── GPU / Device ──────────────────────────────────────────────────────────────
if "{RENDER_ENGINE}" == "CYCLES":
    prefs = bpy.context.preferences.addons["cycles"].preferences
    prefs.compute_device_type = "CUDA"
    prefs.refresh_devices()

    cuda_devices = [d for d in prefs.devices if d.type == "CUDA"]
    if cuda_devices and "{DEVICE}" == "GPU":
        for d in cuda_devices:
            d.use = True
            print("GPU enabled:", d.name)
        scene.cycles.device = "GPU"
    else:
        scene.cycles.device = "CPU"
        print("No CUDA GPU found — rendering on CPU.")

    # Samples
    scene.cycles.samples = {SAMPLES}
    print("Samples set to:", scene.cycles.samples)

    # Adaptive sampling
    scene.cycles.use_adaptive_sampling = {USE_ADAPTIVE_SAMPLING}
    scene.cycles.adaptive_threshold    = {ADAPTIVE_THRESHOLD}
    scene.cycles.adaptive_min_samples  = {MIN_SAMPLES}

    # Denoising
    scene.cycles.use_denoising = {DENOISE}
    if {DENOISE}:
        try:
            scene.cycles.denoiser = "{DENOISE_ALGORITHM}"
        except Exception:
            pass
        try:
            scene.cycles.denoising_input_passes = "{DENOISE_PASS}"
        except Exception:
            pass
        print("Denoising: ON ({DENOISE_ALGORITHM})")
    else:
        print("Denoising: OFF")

# ── Resolution ────────────────────────────────────────────────────────────────
if {RESOLUTION_X} > 0:
    scene.render.resolution_x = {RESOLUTION_X}
if {RESOLUTION_Y} > 0:
    scene.render.resolution_y = {RESOLUTION_Y}
scene.render.resolution_percentage = {RESOLUTION_PERCENT}

# ── Motion blur ───────────────────────────────────────────────────────────────
if "{MOTION_BLUR}" != "None":
    scene.render.use_motion_blur = {MOTION_BLUR}

# ── Output format ─────────────────────────────────────────────────────────────
scene.render.image_settings.file_format  = "{OUTPUT_FORMAT}"
scene.render.image_settings.color_depth  = "{COLOR_DEPTH}"
if "{OUTPUT_FORMAT}" == "JPEG":
    scene.render.image_settings.quality = {OUTPUT_QUALITY}

# ── Output path ───────────────────────────────────────────────────────────────
scene.render.filepath = "{OUTPUT_DIR}frame_####"
print("Output path:", scene.render.filepath)
""")

# ── Render ────────────────────────────────────────────────────────────────────
print(f"🎬 Starting render — version '{CURRENT_VERSION}'")
print(f"   Blend  : {BLEND_FILE}")
print(f"   Frames : {FRAMES_TO_RENDER[0]} → {FRAMES_TO_RENDER[-1]}  ({len(FRAMES_TO_RENDER)} frames)")
print(f"   Engine : {RENDER_ENGINE} on {DEVICE}  |  {SAMPLES} samples")
print("-" * 60)

start_time = time.time()

if RERENDER_FRAMES:
    # Render specific frames one by one
    for frame in FRAMES_TO_RENDER:
        cmd = [BLENDER_BIN, "-b", BLEND_FILE,
               "--python", GPU_SCRIPT,
               "-f", str(frame)]
        proc = subprocess.run(cmd, text=True, capture_output=False)
        if proc.returncode != 0:
            print(f"⚠️  Frame {frame} may have failed (exit code {proc.returncode})")
        else:
            print(f"✅ Frame {frame} done")
else:
    # Render full range
    cmd = [BLENDER_BIN, "-b", BLEND_FILE,
           "--python", GPU_SCRIPT,
           "-s", str(FRAME_START),
           "-e", str(FRAME_END),
           "-j", str(FRAME_STEP),
           "-a"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        # Print lines that show frame progress
        if any(kw in line for kw in ("Fra:", "Saved:", "Finished", "Error", "Warning")):
            print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Blender exited with code {process.returncode}")

elapsed = time.time() - start_time
rendered_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(
    (".png", ".jpg", ".jpeg", ".exr", ".tiff"))])

print("-" * 60)
print(f"✅ Render complete in {elapsed/60:.1f} min  |  {len(rendered_files)} frames saved")

In [ ]:
# ── Cell 9: Save Version Metadata + Package ZIP ──────────────────────────────
import json, os, zipfile, shutil, datetime, pathlib

# ── Save version metadata ─────────────────────────────────────────────────────
versions = load_versions()
version_entry = {
    "label":          CURRENT_VERSION,
    "date":           datetime.datetime.utcnow().isoformat(),
    "blend_file":     BLEND_FILE,
    "start_frame":    FRAME_START,
    "end_frame":      FRAME_END,
    "frame_step":     FRAME_STEP,
    "rerender_frames": RERENDER_FRAMES,
    "engine":         RENDER_ENGINE,
    "device":         DEVICE,
    "samples":        SAMPLES,
    "denoise":        DENOISE,
    "denoise_algo":   DENOISE_ALGORITHM,
    "adaptive":       USE_ADAPTIVE_SAMPLING,
    "resolution":     f"{RESOLUTION_X or 'scene'}x{RESOLUTION_Y or 'scene'}@{RESOLUTION_PERCENT}%",
    "output_format":  OUTPUT_FORMAT,
    "blender_version": BLENDER_VERSION,
    "frames_rendered": len(rendered_files),
    "render_time_min": round(elapsed / 60, 2),
}
# Replace if same label exists, otherwise append
versions = [v for v in versions if v.get("label") != CURRENT_VERSION]
versions.append(version_entry)
VERSION_FILE.write_text(json.dumps(versions, indent=2))
print(f"📝 Version '{CURRENT_VERSION}' saved to history.")

# ── Create ZIP ────────────────────────────────────────────────────────────────
OUTPUT_SAVE_DIR = "/kaggle/working/output/"
os.makedirs(OUTPUT_SAVE_DIR, exist_ok=True)

ZIP_NAME = f"renders_{CURRENT_VERSION}.zip"
ZIP_PATH = f"/kaggle/working/{ZIP_NAME}"

print(f"\n📦 Creating {ZIP_NAME}  ({len(rendered_files)} frames)...")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            if any(file.endswith(ext) for ext in (".png", ".jpg", ".jpeg", ".exr", ".tiff")):
                full_path = os.path.join(root, file)
                zf.write(full_path, arcname=file)

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
final_path = os.path.join(OUTPUT_SAVE_DIR, ZIP_NAME)
shutil.copy(ZIP_PATH, final_path)

print(f"✅ ZIP saved: {final_path}  ({zip_mb:.1f} MB)")
print(f"\n📊 Render Summary")
print(f"   Version       : {CURRENT_VERSION}")
print(f"   Frames        : {len(rendered_files)}")
print(f"   Render time   : {elapsed/60:.1f} min")
print(f"   Blender       : {BLENDER_VERSION}")
print(f"   Samples       : {SAMPLES}  |  Denoise: {DENOISE}")
print(f"   ZIP size      : {zip_mb:.1f} MB")

In [ ]:
# ── Cell 10: Preview & Download Links ────────────────────────────────────────
import os
from IPython.display import display, HTML
from PIL import Image
import base64, io

# ── Thumbnail grid (first 12 frames) ─────────────────────────────────────────
png_files = sorted([
    os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])[:12]

thumbs_html = ""
for fp in png_files:
    try:
        img = Image.open(fp)
        img.thumbnail((160, 90))
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        fname = os.path.basename(fp)
        thumbs_html += (
            f'<div style="display:inline-block;margin:4px;text-align:center">'
            f'<img src="data:image/png;base64,{b64}" style="border:1px solid #444;border-radius:4px"/>'
            f'<br><small style="color:#aaa">{fname}</small></div>'
        )
    except Exception:
        pass

versions = load_versions()
history_rows = "".join(
    f"<tr><td>{v.get('label')}</td><td>{v.get('frames_rendered')}</td>"
    f"<td>{v.get('samples')}</td><td>{v.get('engine')}</td>"
    f"<td>{v.get('render_time_min')} min</td><td>{v.get('blender_version')}</td>"
    f"<td>{v.get('date','')[:10]}</td></tr>"
    for v in reversed(versions)
)

display(HTML(f"""
<style>
  body {{ font-family: sans-serif; background:#111; color:#eee; }}
  h2 {{ color:#f97316; }}
  table {{ border-collapse:collapse; width:100%; margin-top:10px; }}
  th,td {{ border:1px solid #333; padding:6px 10px; text-align:left; font-size:13px; }}
  th {{ background:#222; color:#f97316; }}
  tr:nth-child(even) {{ background:#1a1a1a; }}
  a.dl-btn {{ display:inline-block; background:#f97316; color:#fff; padding:10px 24px;
             border-radius:6px; text-decoration:none; font-weight:bold; margin:8px 4px; }}
  a.dl-btn:hover {{ background:#ea6c0a; }}
</style>

<h2>🎬 RenderFast — {CURRENT_VERSION}</h2>

<p>
  <a class="dl-btn" href="{ZIP_NAME}" target="_blank">⬇ Download {ZIP_NAME}</a>
  <a class="dl-btn" href="output/{ZIP_NAME}" target="_blank">⬇ Alt link (output/)</a>
</p>

<h3 style="color:#ccc">Frame Preview (first {len(png_files)})</h3>
<div style="background:#1a1a1a;padding:8px;border-radius:6px">
{thumbs_html if thumbs_html else "<p style='color:#888'>No PNG frames found for preview.</p>"}
</div>

<h3 style="color:#ccc;margin-top:20px">Render History</h3>
<table>
  <tr><th>Version</th><th>Frames</th><th>Samples</th><th>Engine</th>
      <th>Time</th><th>Blender</th><th>Date</th></tr>
  {history_rows}
</table>
"""))